In [35]:
from transformers import DistilBertTokenizer
from transformers import DistilBertForSequenceClassification ,Trainer, TrainingArguments
import torch
from sklearn.model_selection import train_test_split

import pandas as pd 
import numpy as np

In [36]:
import torch

print(torch.cuda.is_available())

False


In [37]:
tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

In [ ]:
df = pd.read_csv("../data/phishing_email.csv")

df_minmized=df.sample(5000,random_state=42)

df_minmized



,text_combined,label
30381,endangered languages workshop foundation endan...,0
64974,claretta claretta_bordersfusemailcom cialis wo...,1
62961,roger upole schkeramsncom kyle rickey wrote im...,0
81143,barclays customer service testlightworldcojp d...,1
12064,gmm 09 nov 2001 please find attached global ma...,0
...,...,...
62891,timothy murphy sfseoquheircomnet translate pas...,0
68355,tamas sarga tfpigeodtb4gmailcom begin pgp sign...,0
6342,welcome pjm customer info welcome pjm customer...,0
71264,cnn alerts esialnhv_19628wirecouk cnn alerts c...,1


In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    df_minmized["text_combined"],
    df_minmized["label"],
    test_size=0.2,
    random_state=42,
    stratify=df_minmized["label"]
)

print(len(X_train))
print(len(X_test))

4000
1000


In [40]:
train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

In [41]:
class EmailDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels.iloc[idx]
        )

        return item

    def __len__(self):
        return len(self.labels)


train_dataset = EmailDataset(
    train_encodings,
    y_train
)

test_dataset = EmailDataset(
    test_encodings,
    y_test
)

In [45]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [42]:
print(train_dataset[0])

{'input_ids': tensor([  101, 19174, 14771,  7773,  4735,  4638,  8227,  6647,  3246,  2164,
         3668,  2155,  2372,  3647,  2614,  3522, 12642,  4053,  2188, 20033,
         2155,  7118,  3199,  4735,  4638,  6580,  2052,  2612,  2183,  2221,
        26706,  2436,  5958,  2187,  2787,  2175,  6928,  2851,  2349, 12642,
         2145,  1999, 25918,  3085,  3613,  3582,  5791,  2028,  2034,  2240,
         2330, 14771,  2363, 23730,  9131,  3641,  2048,  2197,  2733,  2187,
         2376,  5653,  2015,  2651,  7483,  3433,  2071, 10361,  2092,  3613,
         3046,  2004, 17119, 18249,  3570,  2592, 14771,  7773,  3764,  2957,
         3581, 18133,  2050,  3073,  2566, 10196,  3372, 10373,  2592,  4953,
        19174,  4171,  3314,  4391,  3967,  6647,  4953,  5151,  3980, 28616,
         2278,  2192, 12771,  1023,  2433,  7020,  1018,  2433,  2592,  4663,
         4372,  4948,  2166,  2573,  2047,  4372,  4948,  4268,  2415,  6647,
         2651,  6160,  2621,  2190, 12362,  3960, 

In [43]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    logging_steps=100,
    save_strategy="no"
)


In [46]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)


In [ ]:
trainer.train()


C:\Users\Hassan MS\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
